# Demo — Fitting a Single Observation with the NPE

A worked example of what the trained posterior actually *does*: take one
study's worth of twin covariances and return a full joint posterior over
(A, C, E) in a single forward pass — no optimizer, no per-dataset refit.

Uses the `se_proxy` run from `results/models/`. Figures go to `results/demo/`.

**True parameters:** A = 0.5, C = 0.2, E = 0.3
**Sample size:** N = 1000 twin pairs per zygosity

> This notebook is a demonstration, not a pipeline step — it does not feed
> STEP 06. See README.md for the numbered pipeline.


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ace_model.py is the shared library: model math, paths, posterior helpers.
sys.path.insert(0, str(Path.cwd()))
from ace_model import (
    ACE_PARAM_NAMES,
    ACEEmbeddingNet,   # noqa: F401 - importable so pickle can rebuild a posterior
    DEMO_DIR,
    MODELS_DIR,
    build_features,
    load_posterior,
    simulate_covariances,
    summarize_cov,
)

DEMO_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')


In [ ]:
# -- True parameters & sample size ------------------------------------------
A_TRUE, C_TRUE, E_TRUE = 0.5, 0.2, 0.3
N_PAIRS = 1000
SEED = 42
MODEL_DIR = MODELS_DIR / 'se_proxy'

np.random.seed(SEED)

# -- Simulate one observed pair of covariance matrices ----------------------
S_mz, S_dz = simulate_covariances(A_TRUE, C_TRUE, E_TRUE, N_pairs=N_PAIRS)

# summarize_cov applies the sufficient reduction: the *_var feature is the
# MEAN of the two diagonal entries, since both estimate the same phenotypic
# variance. Using only S[0,0] would discard half the information about it.
mz_var, mz_cov = summarize_cov(S_mz)
dz_var, dz_cov = summarize_cov(S_dz)

print(f'True:  A={A_TRUE:.2f}  C={C_TRUE:.2f}  E={E_TRUE:.2f}')
print(f'N pairs: {N_PAIRS}')
print('\nSimulated statistics (vs. their theoretical values):')
print(f'  mz_var = {mz_var:.4f}   (theory: {A_TRUE+C_TRUE+E_TRUE:.4f})')
print(f'  mz_cov = {mz_cov:.4f}   (theory: {A_TRUE+C_TRUE:.4f})')
print(f'  dz_var = {dz_var:.4f}   (theory: {A_TRUE+C_TRUE+E_TRUE:.4f})')
print(f'  dz_cov = {dz_cov:.4f}   (theory: {0.5*A_TRUE+C_TRUE:.4f})')


In [ ]:
# -- Load the trained posterior ---------------------------------------------
loaded       = load_posterior(MODEL_DIR)
posterior    = loaded['posterior']
scaler       = loaded['scaler']
feature_cols = loaded['feature_cols']

print(f'Model dir:     {MODEL_DIR.name}')
print(f'Feature cols:  {feature_cols}')
print('Posterior loaded successfully.')


In [ ]:
# -- Scale the observation and draw posterior samples -----------------------
N_POSTERIOR = 5000

# build_features appends the N-derived feature in whatever encoding this run
# was trained with (here: se_proxy = 1/sqrt(N)), in the right column order.
x_raw    = build_features(mz_var, mz_cov, dz_var, dz_cov, N_PAIRS, feature_cols)
x_scaled = scaler.transform(x_raw).flatten()
x_tensor = torch.FloatTensor(x_scaled).unsqueeze(0)

with torch.no_grad():
    samples = posterior.sample(
        (N_POSTERIOR,), x=x_tensor,
        show_progress_bars=False,
        reject_outside_prior=False,
    )

samples_np = samples.numpy()   # (N_POSTERIOR, 3)

post_mean = samples_np.mean(axis=0)
post_std  = samples_np.std(axis=0)
post_med  = np.median(samples_np, axis=0)

print(f'Posterior draws: {N_POSTERIOR}')
print(f'\n{"":<6} {"True":>8} {"Mean":>8} {"Median":>8} {"SD":>8}')
print('-' * 44)
for i, p in enumerate(ACE_PARAM_NAMES):
    truth = [A_TRUE, C_TRUE, E_TRUE][i]
    print(f'{p:<6} {truth:>8.4f} {post_mean[i]:>8.4f} {post_med[i]:>8.4f} {post_std[i]:>8.4f}')


In [ ]:
# -- Marginal posteriors ----------------------------------------------------
colors = ['#2196F3', '#4CAF50', '#FF9800']

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for i, (ax, param, color) in enumerate(zip(axes, ACE_PARAM_NAMES, colors)):
    ax.hist(samples_np[:, i], bins=60, color=color, alpha=0.75,
            density=True, edgecolor='white')
    ax.axvline([A_TRUE, C_TRUE, E_TRUE][i], color='black', lw=2,
               linestyle='-', label='True')
    ax.axvline(post_mean[i], color='red',  lw=1.5, linestyle='--',
               label=f'Mean = {post_mean[i]:.3f}')
    ax.axvline(post_med[i],  color='navy', lw=1.5, linestyle=':',
               label=f'Median = {post_med[i]:.3f}')
    ax.set_xlabel(param, fontsize=12)
    ax.set_ylabel('Density' if i == 0 else '', fontsize=11)
    ax.set_title(f'{param}  (sigma = {post_std[i]:.3f})', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f'NPE Posterior  |  MZ: var={mz_var:.3f}, cov={mz_cov:.3f}  '
    f'DZ: var={dz_var:.3f}, cov={dz_cov:.3f}  |  N={N_PAIRS} pairs',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
out = DEMO_DIR / 'demo_posterior.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {out}')


In [ ]:
# -- 2-D joint posteriors ---------------------------------------------------
# The ridges here are the point: A, C and E are only jointly identified, and
# the flow captures that correlation structure directly.
pairs = [('A', 'C', 0, 1), ('A', 'E', 0, 2), ('C', 'E', 1, 2)]
truth = {'A': A_TRUE, 'C': C_TRUE, 'E': E_TRUE}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (px_name, py_name, ix, iy) in zip(axes, pairs):
    ax.hexbin(samples_np[:, ix], samples_np[:, iy],
              gridsize=40, cmap='Blues', mincnt=1)
    ax.axvline(truth[px_name], color='red', lw=1.5, linestyle='--')
    ax.axhline(truth[py_name], color='red', lw=1.5, linestyle='--', label='True')
    ax.set_xlabel(px_name, fontsize=11)
    ax.set_ylabel(py_name, fontsize=11)
    ax.set_title(f'{px_name} vs {py_name}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

fig.suptitle('Joint Posterior Densities', fontsize=12, fontweight='bold')
plt.tight_layout()
out = DEMO_DIR / 'demo_joint_posterior.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {out}')
